In [14]:
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime
from pathlib import Path

In [32]:
#read chronos file
def time_reader(f):
    f=open(f)
    lines = f.readlines()
    time_map = {} 
    #start datetime: {end datetime: e_dt, network latency per img: nw, 
    # compute latency per imd : comp, throughput: th, batch_size: bs, num_batches: nb, 
    # total_time: e_dt - start_dt, (apply day correction if required) 
    # network latency total: nw_t, compute latency total: comp_t, 
    # flop: fp, s_per_op: s*10**9/op, nw_std: 0, comp_std: 0 } 
    dt_st=0
    dt_et=0
    for line in lines:
        line=line.strip()
        if "startheat" in line:
            st = line.split("TIME:")[-1]
            dt_st = datetime.strptime(st, "%H:%M:%S.%f")
        if "FLOP count" in line:
            #this means it was successful (probably)
            #create dict object here
            # rank 0 FLOP count: 472055808 and full time*10**9 s/op: 0.3508
            flop = line.split("count: ")[-1].split(" and ")[0]
            s_per_op = line.split("s/op: ")[-1]
            time_map[dt_st] = {"e_dt":0, "nw":0, "comp":0, "th":0, "bs":0, "nb":0, "total_time":0, 
            "nw_t":0, "comp_t":0, "flop":int(flop), "s_per_op": float(s_per_op), "nw_std": 0, "comp_std": 0}
        elif "Time taken by rank:" in line:
            # Time taken by rank:0 in total(avg): 0.3312s on avg per image: 0.1656s with std: 0.0050s network time: 0.0798s 
            # and network std: 0.0798s network time per img: 0.0399 compute time: 0.1257s 
            # compute/network ratio: 1.5764 and throughput 6.0381 img/s
            nw_t = line.split("network time: ")[-1].split("s")[0]
            nw = line.split("network time per img: ")[-1].split("s")[0]
            nw_std = line.split("network std: ")[-1].split("s")[0]
            comp_t = line.split("total(avg): ")[-1].split("s")[0]
            comp = line.split("avg per image: ")[-1].split("s")[0]
            comp_std = line.split("with std: ")[-1].split("s")[0]
            th = line.split("throughput ")[-1].split(" ")[0]
            time_map[dt_st]["nw_t"] = nw_t
            time_map[dt_st]["nw"] =  nw
            time_map[dt_st]["nw_std"]=nw_std
            time_map[dt_st]["comp_t"] =comp_t
            time_map[dt_st]["comp"] = comp
            time_map[dt_st]["comp_std"]=comp_std
            time_map[dt_st]["th"]=th
        if "endheat" in line:
            et  = line.split("TIME:")[-1]
            if dt_st not in time_map:
                break
            dt_et = datetime.strptime(et, "%H:%M:%S.%f")
            time_map[dt_st]["e_dt"] = dt_et
    
    if dt_et==0 or dt_st==0:
        return None

         
    return time_map

#read heat file
def heat_reader(f):
    pass
#time sync/match chronos and heat file


In [33]:
failure = {}
world_size_count = 0
world_size_5_count = 0
world_size_5_fail_count = 0
num_batch_size_10_count = 0
num_batch_size_10_fail_count = 0
batch_size_6_count = 0
batch_size_6_fail_count = 0
total_counter=0
for mid in range(1,42): #device id in subcluster
    set_flag=False
    for world_size in range(1,6): #number of layer splits/pipeline depth
        for batch_size in [1,4,6]:  #number of images in a batch per inference
            for num_batch in [1,10]: #number of batches per exp
                counter=0
                # failures = 
                #try to sweep all possible files:
                for trial in range(1,42):
                    fname = f"../logs/sweep/b-1-2/1_ranking_logs/{mid}_{world_size}_{batch_size}_{num_batch}_logs/speed_chronosbramble-1-2-{mid}.log"
                    if Path(fname).is_file():
                        set_flag=True
                        # print(mid, world_size, batch_size, num_batch)
                        time_map = time_reader(fname)
                        total_counter+=1
                        # print(time_map!=None)
                        if world_size==5:
                            world_size_5_count+=1
                        if num_batch==10:
                            num_batch_size_10_count+=1
                        if batch_size==6:
                            batch_size_6_count+=1
                        if time_map==None:
                            failure[f"main_{mid}_{world_size}_{batch_size}_{num_batch}_worker_{trial}"] = (time_map==None)
                            if world_size==5:
                                world_size_5_fail_count+=1
                            if num_batch==10:
                                num_batch_size_10_fail_count+=1
                            if batch_size==6:
                                batch_size_6_fail_count+=1
                        counter+=1
                    if counter==world_size:
                        break
    # if set_flag:
    #     break
print(failure.keys())
print(world_size_5_count, world_size_5_fail_count)
print(num_batch_size_10_count, num_batch_size_10_fail_count)
print(batch_size_6_count, batch_size_6_fail_count)
print(len(failure))
print(total_counter-len(failure))


# time_reader("../logs/sweep/b-1-2/1_ranking_logs/13_2_6_10_logs/speed_chronosbramble-1-2-13.log")

dict_keys(['main_2_2_6_10_worker_1', 'main_2_2_6_10_worker_2', 'main_2_5_1_10_worker_1', 'main_2_5_1_10_worker_2', 'main_2_5_1_10_worker_3', 'main_2_5_1_10_worker_4', 'main_2_5_1_10_worker_5', 'main_2_5_4_10_worker_1', 'main_2_5_4_10_worker_2', 'main_2_5_4_10_worker_3', 'main_2_5_4_10_worker_4', 'main_2_5_4_10_worker_5', 'main_2_5_6_10_worker_1', 'main_2_5_6_10_worker_2', 'main_2_5_6_10_worker_3', 'main_2_5_6_10_worker_4', 'main_2_5_6_10_worker_5', 'main_4_2_6_10_worker_1', 'main_4_2_6_10_worker_2', 'main_5_1_6_10_worker_1', 'main_5_5_1_1_worker_1', 'main_5_5_1_1_worker_2', 'main_5_5_1_1_worker_3', 'main_5_5_1_1_worker_4', 'main_5_5_1_1_worker_5', 'main_5_5_4_10_worker_1', 'main_5_5_4_10_worker_2', 'main_5_5_4_10_worker_3', 'main_5_5_4_10_worker_4', 'main_5_5_4_10_worker_5', 'main_7_1_6_10_worker_1', 'main_7_5_4_1_worker_1', 'main_7_5_4_1_worker_2', 'main_7_5_4_1_worker_3', 'main_7_5_4_1_worker_4', 'main_7_5_4_1_worker_5', 'main_7_5_4_10_worker_1', 'main_7_5_4_10_worker_2', 'main_7_5_4